# Pandas Overview

Before writing pandas code, we answer five questions: **what, why, when, where, how**.
This notebook is the big picture of the full pandas chapter. Read it once now, and again before interviews.

**Contents**
1. What is pandas?
2. Why pandas after NumPy? (with live proof)
3. The two core data structures
4. When to use pandas (and when not)
5. Where pandas is used (AI/ML focus)
6. How pandas works: the 6-step workflow
7. Why pandas is fast (with timing proof)
8. NumPy vs pandas
9. Pandas in the ML pipeline
10. Chapter roadmap
11. Version notes (pandas 3.x)
12. Quick taste
13. Project preview
14. Interview questions
15. Summary

In [1]:
import numpy as np
import pandas as pd

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.6
numpy : 2.5.3


---
# 1. What is pandas?

- Pandas is a Python library for working with **table-like data**: rows and columns, like an Excel sheet, a SQL table or a CSV file.
- The name comes from **"Panel Data"** (an economics term), not from the animal.
- It is built **on top of NumPy**. Every pandas column is a NumPy array inside.
- Created by **Wes McKinney in 2008** because NumPy alone was painful for real, messy data.

> **One line:**
> NumPy = fast math on numbers.
> Pandas = NumPy + labels + mixed data types + missing data handling + file reading.

---
# 2. Why pandas after NumPy?

This is the most important question of this notebook. NumPy is great, but real-world data breaks its limits.
Real data looks like this:

| name | age | city | salary | joined_on |
|---|---|---|---|---|
| Ravi | 21 | Indore | 50000 | 2024-01-10 |
| Asha | 22 | *(blank)* | 62000 | 2024-03-05 |

It has **text, numbers, dates and blank cells together**. Let us see how NumPy struggles, and then how pandas solves each problem.

## Problem 1: NumPy allows only ONE data type per array

In [2]:
row = np.array(["Ravi", 21, 50000.5])
print(row)
print(row.dtype)      # everything became text!

['Ravi' '21' '50000.5']
<U32


NumPy converted **everything into strings**. Now `21` is not a number anymore, so we cannot do maths on it.
Pandas gives **each column its own dtype**:

In [3]:
df = pd.DataFrame({
    "name": ["Ravi", "Asha"],
    "age": [21, 22],
    "salary": [50000.5, 62000.0],
})
print(df.dtypes)
print()
print(df["age"].mean())    # works, because age is still a number

name          str
age         int64
salary    float64
dtype: object

21.5


## Problem 2: NumPy has no labels, only positions

In [4]:
arr = np.array([50000, 62000, 58000])
print(arr[1])      # whose salary is this? I have to remember that index 1 = Asha

62000


In [5]:
salary = pd.Series([50000, 62000, 58000], index=["Ravi", "Asha", "Karan"])
print(salary["Asha"])    # readable, no need to remember positions

62000


## Problem 3: Missing values break NumPy maths

In [6]:
a = np.array([10.0, np.nan, 30.0])
print(np.mean(a))        # nan -> one missing value ruins the result

nan


In [7]:
s = pd.Series([10.0, np.nan, 30.0])
print(s.mean())          # pandas skips NaN automatically
print(s.fillna(s.mean()).tolist())    # and gives easy tools to fill/drop them

20.0
[10.0, 20.0, 30.0]


## Problem 4: NumPy cannot read real files properly

Pandas reads a CSV in one line. (Here we use a fake file inside a string, the next chapter uses real files.)

In [8]:
from io import StringIO

fake_csv = StringIO("""name,age,city,salary
Ravi,21,Indore,50000
Asha,22,,62000
Karan,23,Delhi,58000
""")

data = pd.read_csv(fake_csv)
data

,name,age,city,salary
0,Ravi,21,Indore,50000
1,Asha,22,NaN,62000
2,Karan,23,Delhi,58000


Blank cell in `city` became `NaN` automatically, and `age`, `salary` became numbers. NumPy cannot do this.

## Problem 5: No group-by, join, pivot, date tools in NumPy

Pandas gives these for free. Here is a small taste of **groupby** (one line to get average score per city):

In [9]:
scores = pd.DataFrame({
    "city":  ["Indore", "Delhi", "Indore", "Delhi", "Pune"],
    "score": [80, 70, 90, 60, 75],
})
scores.groupby("city")["score"].mean()

city
Delhi     65.0
Indore    85.0
Pune      75.0
Name: score, dtype: float64

## Summary table: NumPy limit -> pandas fix

| Problem in NumPy | How pandas fixes it |
|---|---|
| One dtype per array | Each column has its own dtype |
| Access only by position | Access by label: `df["salary"]`, `df.loc["Ravi"]` |
| No column names | Rows and columns both have names |
| Weak missing-value support | `isna`, `fillna`, `dropna` |
| No file reading | `read_csv`, `read_excel`, `read_json`, `read_sql` |
| No group-by / join / pivot | `groupby`, `merge`, `pivot_table` |
| No date/time tools | Strong datetime support |

**Natural learning order:** Python -> NumPy (fast numbers) -> **Pandas (real tables)** -> Matplotlib/Seaborn (plots) -> scikit-learn (ML) -> Deep Learning

---
# 3. The two core data structures

| Structure | Dimension | Meaning |
|---|---|---|
| **Series** | 1D | One column (a NumPy array + index labels) |
| **DataFrame** | 2D | Full table (many Series sharing one index) |

```
            DataFrame
 index |  name  | age | city        <- each column is a Series
   0   |  Ravi  | 21  | Indore
   1   |  Asha  | 22  | Bhopal
```

In AI/ML: **DataFrame = your dataset**, **one Series = one feature (or the target)**.

Next notebook (`02_series_and_dataframe.ipynb`) covers both in detail.

In [10]:
age = pd.Series([21, 22, 23], name="age")
table = pd.DataFrame({"age": age, "score": [80, 90, 70]})

print(type(age))
print(type(table))
print(type(table["score"]))     # a column of a DataFrame is a Series

<class 'pandas.Series'>
<class 'pandas.DataFrame'>
<class 'pandas.Series'>


---
# 4. When to use pandas (and when not)

### Use pandas when
- Data is **tabular** (CSV, Excel, SQL table, JSON list from an API)
- You need to **clean** data (missing values, duplicates, wrong types)
- You need to **filter, group, join, sort, summarize**
- Data **fits in RAM** (up to a few GB is comfortable)
- You are doing **EDA** (exploring data) before ML or reporting

### Do NOT use pandas when
| Situation | Better tool |
|---|---|
| Pure number crunching, matrices, images | NumPy |
| Data much bigger than RAM (100 GB+) | Spark, Polars, DuckDB, SQL |
| Deep learning tensors | PyTorch / TensorFlow |
| Real-time row-by-row streaming | Plain Python / stream tools |

> **Rule of thumb:** "Does this look like a spreadsheet?" -> use pandas.

---
# 5. Where pandas is used (AI/ML focus)

**In AI / ML / Data Science**
- Load and explore a dataset (EDA) before training any model
- Clean data: missing values, duplicates, outliers, wrong dtypes
- **Feature engineering**: create new useful columns from old ones
- Encode categories (like `city`) into numbers for models
- Split into **X (features)** and **y (target)** for scikit-learn / PyTorch
- Analyze model results: predictions vs actual, errors per category

**In other fields**
- Finance: stock prices, time series
- Business: sales reports, customer analytics
- Backend / APIs: processing JSON data
- Automation: merging many Excel/CSV files into one report

> Fun fact: in a real ML project, **60-80% of the time goes into data preparation**, and that is pandas work. A mediocre model on clean data beats a great model on dirty data.

**For interviews:** pandas is not asked in DSA rounds, but it is very common in **ML Engineer, Data Scientist, Data Engineer and Analytics** rounds (usually as: "given this DataFrame, write code to find X").

---
# 6. How pandas works: the 6-step workflow

Every pandas project follows the same flow:

```
 1. LOAD  ->  2. INSPECT  ->  3. CLEAN  ->  4. TRANSFORM  ->  5. ANALYZE  ->  6. EXPORT
```

| Step | What we do | Common methods |
|---|---|---|
| 1. Load | Read data into a DataFrame | `read_csv`, `read_excel`, `read_json`, `read_sql` |
| 2. Inspect | Understand the data | `head`, `info`, `describe`, `shape`, `dtypes` |
| 3. Clean | Fix problems | `isna`, `fillna`, `dropna`, `drop_duplicates`, `astype`, `rename` |
| 4. Transform | Create/shape new data | new columns, filtering, `apply`, `map` |
| 5. Analyze | Find answers | `groupby`, `merge`, `pivot_table`, `corr` |
| 6. Export | Save or pass on | `to_csv`, `to_excel`, `to_sql`, plots, `to_numpy()` for the model |

Let us run all 6 steps on a tiny dataset:

In [11]:
from io import StringIO

raw = StringIO("""student,hours,attendance,score
Ravi,5.5,80,85
Asha,7,90,92
Karan,3,,58
Neha,6.5,85,88
Karan,3,,58
Amit,4,70,64
""")

# 1. LOAD
df = pd.read_csv(raw)

# 2. INSPECT
print(df.shape)
df.info()

(6, 4)
<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   student     6 non-null      str    
 1   hours       6 non-null      float64
 2   attendance  4 non-null      float64
 3   score       6 non-null      int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 324.0 bytes


In [12]:
# 3. CLEAN: remove duplicate row, fill missing attendance with median
df = df.drop_duplicates()
df["attendance"] = df["attendance"].fillna(df["attendance"].median())
df

,student,hours,attendance,score
0,Ravi,5.5,80.0,85
1,Asha,7.0,90.0,92
2,Karan,3.0,82.5,58
3,Neha,6.5,85.0,88
5,Amit,4.0,70.0,64


In [13]:
# 4. TRANSFORM: create a new column (feature engineering)
df["passed"] = df["score"] >= 60
df["score_per_hour"] = (df["score"] / df["hours"]).round(2)
df

,student,hours,attendance,score,passed,score_per_hour
0,Ravi,5.5,80.0,85,True,15.45
1,Asha,7.0,90.0,92,True,13.14
2,Karan,3.0,82.5,58,False,19.33
3,Neha,6.5,85.0,88,True,13.54
5,Amit,4.0,70.0,64,True,16.00


In [14]:
# 5. ANALYZE: does studying more give more marks?
print(df[["hours", "attendance", "score"]].corr()["score"])

hours         0.981370
attendance    0.625881
score         1.000000
Name: score, dtype: float64


In [15]:
# 6. EXPORT: prepare model input (X, y) as NumPy arrays
X = df[["hours", "attendance"]].to_numpy()
y = df["score"].to_numpy()
print(X.shape, y.shape)
# df.to_csv("clean_students.csv", index=False)    # saving to a file

(5, 2) (5,)


---
# 7. Why pandas is fast

- Each column is stored as a **NumPy array** (continuous memory, C code underneath).
- Operations are **vectorized**: one operation is applied to the whole column at once, no Python loop.

> **Golden rule:** if you are writing a `for` loop over rows in pandas, stop and look for a vectorized method first.

Proof (timing a loop vs vectorized on 200,000 rows):

In [16]:
import time

big = pd.DataFrame({
    "price": np.random.rand(200_000) * 100,
    "qty": np.random.randint(1, 10, 200_000),
})

# Slow way: Python loop
start = time.perf_counter()
total_loop = []
for i in range(len(big)):
    total_loop.append(big["price"].iloc[i] * big["qty"].iloc[i])
loop_time = time.perf_counter() - start

# Fast way: vectorized
start = time.perf_counter()
total_vec = big["price"] * big["qty"]
vec_time = time.perf_counter() - start

print(f"loop      : {loop_time:.4f} sec")
print(f"vectorized: {vec_time:.6f} sec")
print(f"vectorized is about {loop_time / vec_time:,.0f}x faster")

loop      : 3.3403 sec
vectorized: 0.000417 sec
vectorized is about 8,006x faster


---
# 8. NumPy vs pandas

| Feature | NumPy | pandas |
|---|---|---|
| Main object | `ndarray` | `Series`, `DataFrame` |
| Dimensions | N-dimensional | 1D and 2D |
| Data types | One per array | One per column |
| Indexing | Position only | Position + labels |
| Missing values | Weak | Strong |
| File reading | Very basic | CSV, Excel, SQL, JSON, Parquet |
| Alignment in operations | By position | By label |
| Best for | Maths, matrices, images, speed | Cleaning and analysing tables |
| Speed | Faster for raw numbers | Slight overhead, many more features |

They are **partners, not rivals**: pandas for preparing data, NumPy for the maths part.

---
# 9. Pandas in the ML pipeline

```
 Raw data (CSV / DB / API)
        |
        v
 [ pandas ]  load -> inspect -> clean -> feature engineering
        |
        v
 [ NumPy ]   X, y arrays (fast numeric form)
        |
        v
 [ scikit-learn / PyTorch ]  train model
        |
        v
 [ pandas ]  analyze predictions, errors, report results
        |
        v
 [ Matplotlib / Seaborn ]  charts
```

Pandas is at the **start and the end** of almost every ML project.

---
# 10. Chapter roadmap

| No. | Topic |
|---|---|
| 01 | Overview|
| 02 | Series and DataFrame|
| 03 | Reading and writing data (CSV, Excel, JSON, SQL) |
| 04 | Inspecting data (`head`, `info`, `describe`, `value_counts`) |
| 05 | Selecting data (`loc`, `iloc`, boolean filtering, `query`) |
| 06 | Adding, updating, dropping columns and rows |
| 07 | Handling missing data |
| 08 | Data cleaning (duplicates, dtypes, string methods) |
| 09 | Sorting, ranking, `apply` / `map` |
| 10 | GroupBy and aggregation |
| 11 | Merge, join, concat |
| 12 | Pivot tables, crosstab, reshaping (`melt`, `stack`) |
| 13 | Datetime and time series |
| 14 | Performance and best practices |
| 15 | Visualization basics |
| 16 | **Final project** (real-world AI/ML problem) |

---
# 11. Version notes (pandas 3.x)

My setup: **pandas 3.0.6, numpy 2.5.3, Python 3.12.7 (.venv)**.

Pandas 3.x is a big release. Old tutorials online may behave differently:
- **Copy-on-Write is the default.** Changing a subset never silently changes the original DataFrame.
- **Text columns use a dedicated string dtype** (shown as `str`) instead of `object`.
- Avoid `inplace=True`. Assign results back instead (`df = df.drop(...)`).

Always trust the official docs for the version you installed.

---
# 12. Quick taste

Small things you will be able to do after this chapter:

In [17]:
people = pd.DataFrame({
    "name": ["Ravi", "Asha", "Karan", "Neha"],
    "age": [21, 22, 23, 21],
    "city": ["Indore", "Bhopal", "Delhi", "Indore"],
    "salary": [50000, 62000, 58000, 55000],
})

print(people[people["salary"] > 55000])      # filter in one line
print()
print(people["salary"].mean())               # quick stats
print()
print(people["city"].value_counts())         # frequency count
print()
print(people.sort_values("salary", ascending=False).head(2))

    name  age    city  salary
1   Asha   22  Bhopal   62000
2  Karan   23   Delhi   58000

56250.0

city
Indore    2
Bhopal    1
Delhi     1
Name: count, dtype: int64

    name  age    city  salary
1   Asha   22  Bhopal   62000
2  Karan   23   Delhi   58000


---
# 13. Project preview

At the end we will build a **real-world AI/ML project.

Idea options (we will finalise after a few chapters):
- **Student Performance / Dropout Predictor**: clean student data, find which factors matter, train a simple model
- **Used Car Price Analyzer**: messy listings data -> cleaning -> feature engineering -> price insights
- **Customer Churn Explorer**: find which customers are likely to leave and why

The project will show the full flow: **raw messy data -> pandas cleaning and analysis -> model -> insights**, which is exactly what ML interviewers want to see.

---
# 14. Interview questions

**Q1. Why do we use pandas when NumPy already exists?**
NumPy has one dtype per array and only positional access. Pandas adds labels, different dtype per column, missing-value handling, file reading, groupby, merge and time-series tools.

**Q2. What are the two main data structures of pandas?**
Series (1D labeled) and DataFrame (2D labeled table).

**Q3. Why is pandas fast?**
Columns are NumPy arrays underneath and operations are vectorized in C, so there is no Python loop.

**Q4. When would you NOT use pandas?**
When the data is too big for RAM (use Spark / Polars / DuckDB), or when you only need numeric array maths (use NumPy), or for deep learning tensors.

**Q5. What is the typical pandas workflow?**
Load -> Inspect -> Clean -> Transform -> Analyze -> Export.

**Q6. Which is faster for pure numeric computation, NumPy or pandas?**
NumPy, because pandas adds overhead for labels and alignment.

**Q7. Where does pandas fit in an ML pipeline?**
At the start (loading, cleaning, feature engineering) and the end (analysing predictions and errors). The model itself uses NumPy arrays or tensors.

---
# 15. Summary

- Pandas = **NumPy + labels + mixed types + missing data + file I/O + groupby/merge**
- Two objects: **Series (1D)** and **DataFrame (2D)**
- Workflow: **Load -> Inspect -> Clean -> Transform -> Analyze -> Export**
- Avoid row loops, use **vectorized** operations
- In ML, pandas **prepares the data**, NumPy and the ML library **use it**
- Do not use pandas for data bigger than RAM or for pure matrix maths